In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
import os
if not os.path.exists('/content/cdt-alzheimer-screening'):
    !git clone https://github.com/wiambenadder/cdt-alzheimer-screening.git
%cd cdt-alzheimer-screening
!git pull 2>/dev/null
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, '/content/cdt-alzheimer-screening')
print("setup ok")

Mounted at /content/drive
/content
Cloning into 'cdt-alzheimer-screening'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 110 (delta 48), reused 29 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 643.21 KiB | 4.80 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/cdt-alzheimer-screening
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 148.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 75.2 MB/s eta 0:00:00
setup 

In [2]:
%%writefile /content/cdt-alzheimer-screening/src/data.py
"""NHATS CDT dataset: loading, splits, class imbalance, dataloaders."""
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

from .config import DataConfig, SEED, NUM_CLASSES, LABELS_CSV, RAW_IMAGES_DIR


class CDTDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, int(row["label"])


def load_labels(labels_csv=LABELS_CSV, images_dir=RAW_IMAGES_DIR,
                image_ext=".tif", score_column="cdt_score",
                id_column="participant_id"):
    df = pd.read_csv(labels_csv)
    df = df.rename(columns={score_column: "label", id_column: "participant_id"})
    if "image_path" not in df.columns:
        df["image_path"] = df["participant_id"].apply(
            lambda pid: str(images_dir / f"{pid}{image_ext}"))
    df = df[df["label"].between(0, NUM_CLASSES - 1)].copy()
    df["label"] = df["label"].astype(int)
    exists_mask = df["image_path"].apply(lambda p: Path(p).exists())
    if (~exists_mask).sum():
        print(f"[data] dropped {(~exists_mask).sum()} rows with missing files")
    df = df[exists_mask].reset_index(drop=True)
    print(f"[data] loaded {len(df)} labeled clock images")
    return df


def stratified_split(df, cfg):
    train_df, temp_df = train_test_split(
        df, test_size=1 - cfg.train_ratio, stratify=df["label"], random_state=SEED)
    val_size = cfg.val_ratio / (cfg.val_ratio + cfg.test_ratio)
    val_df, test_df = train_test_split(
        temp_df, train_size=val_size, stratify=temp_df["label"], random_state=SEED)
    print(f"[split] train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def participant_disjoint_split(df, cfg):
    """Leak-safe split: every participant appears in exactly one split."""
    gss1 = GroupShuffleSplit(n_splits=1, train_size=cfg.train_ratio, random_state=SEED)
    train_idx, temp_idx = next(gss1.split(df, groups=df["participant_id"]))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    temp_df  = df.iloc[temp_idx].reset_index(drop=True)

    val_ratio_in_temp = cfg.val_ratio / (cfg.val_ratio + cfg.test_ratio)
    gss2 = GroupShuffleSplit(n_splits=1, train_size=val_ratio_in_temp, random_state=SEED)
    val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["participant_id"]))
    val_df  = temp_df.iloc[val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)

    train_ids = set(train_df["participant_id"])
    val_ids   = set(val_df["participant_id"])
    test_ids  = set(test_df["participant_id"])
    assert not (train_ids & val_ids), "train/val leak"
    assert not (train_ids & test_ids), "train/test leak"
    assert not (val_ids & test_ids), "val/test leak"

    print(f"[split] participant-disjoint:")
    print(f"  train: {len(train_df):,} clocks from {len(train_ids):,} participants")
    print(f"  val:   {len(val_df):,} clocks from {len(val_ids):,} participants")
    print(f"  test:  {len(test_df):,} clocks from {len(test_ids):,} participants")
    return train_df, val_df, test_df


def compute_class_weights(train_df):
    labels = train_df["label"].values
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    full = np.ones(NUM_CLASSES, dtype=np.float32)
    for c, w in zip(classes, weights):
        full[c] = w
    return torch.tensor(full, dtype=torch.float32)


def make_weighted_sampler(train_df):
    counts = train_df["label"].value_counts().sort_index()
    sw = train_df["label"].apply(lambda y: 1.0 / counts[y]).values
    return WeightedRandomSampler(weights=sw, num_samples=len(sw), replacement=True)


def build_dataloaders(train_ds, val_ds, test_ds, cfg, sampler=None):
    shuffle_train = sampler is None
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=shuffle_train, sampler=sampler,
        num_workers=cfg.num_workers, pin_memory=True, drop_last=True,
        persistent_workers=(cfg.num_workers > 0))
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True,
        persistent_workers=(cfg.num_workers > 0))
    test_loader = DataLoader(
        test_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True,
        persistent_workers=(cfg.num_workers > 0))
    return train_loader, val_loader, test_loader

Overwriting /content/cdt-alzheimer-screening/src/data.py


In [4]:
%%writefile /content/cdt-alzheimer-screening/src/train.py
"""Training loop with AMP, schedulers, early stopping, TensorBoard."""
import json
import os
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
from torch.optim import AdamW, Adam, SGD
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

from .config import TrainConfig, MODELS_DIR, RESULTS_DIR
from .models import get_model
from .utils import set_seed, save_checkpoint, get_device, count_parameters


def build_optimizer(backbone_params, head_params, cfg: TrainConfig):
    param_groups = []
    if len(backbone_params) > 0:
        param_groups.append({"params": backbone_params, "lr": cfg.lr_backbone})
    if len(head_params) > 0:
        param_groups.append({"params": head_params, "lr": cfg.lr_head})
    if cfg.optimizer == "adamw":
        return AdamW(param_groups, weight_decay=cfg.weight_decay)
    if cfg.optimizer == "adam":
        return Adam(param_groups, weight_decay=cfg.weight_decay)
    if cfg.optimizer == "sgd":
        return SGD(param_groups, momentum=0.9, weight_decay=cfg.weight_decay)
    raise ValueError(f"Unknown optimizer {cfg.optimizer!r}")


def build_scheduler(optimizer, cfg: TrainConfig, steps_per_epoch: int):
    if cfg.lr_scheduler == "cosine":
        return CosineAnnealingLR(optimizer, T_max=cfg.epochs * steps_per_epoch)
    if cfg.lr_scheduler == "plateau":
        return ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)
    if cfg.lr_scheduler == "none":
        return None
    raise ValueError(f"Unknown scheduler {cfg.lr_scheduler!r}")


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler,
                    device, cfg: TrainConfig, epoch: int, writer=None):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"epoch {epoch} [train]", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast(enabled=cfg.mixed_precision):
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None and not isinstance(scheduler, ReduceLROnPlateau):
            scheduler.step()
        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f"{running_loss/total:.4f}", acc=f"{correct/total:.3f}")
    tr_loss = running_loss / total
    tr_acc = correct / total
    if writer is not None:
        writer.add_scalar("train/loss", tr_loss, epoch)
        writer.add_scalar("train/acc", tr_acc, epoch)
        for i, pg in enumerate(optimizer.param_groups):
            writer.add_scalar(f"train/lr_group_{i}", pg["lr"], epoch)
    return tr_loss, tr_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch=None, writer=None, tag="val"):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    ev_loss = running_loss / total
    ev_acc = correct / total
    if writer is not None and epoch is not None:
        writer.add_scalar(f"{tag}/loss", ev_loss, epoch)
        writer.add_scalar(f"{tag}/acc", ev_acc, epoch)
    return ev_loss, ev_acc


def train(model_name, train_loader, val_loader, class_weights, cfg, run_name,
          freeze_backbone=False):
    set_seed()
    device = get_device()
    print(f"[train] run={run_name} device={device}")

    model, backbone_params, head_params = get_model(
        model_name, freeze_backbone=freeze_backbone, dropout=cfg.dropout,
    )
    model = model.to(device)
    total, trainable = count_parameters(model)
    print(f"[train] params total={total:,} trainable={trainable:,}")

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = build_optimizer(backbone_params, head_params, cfg)
    scheduler = build_scheduler(optimizer, cfg, len(train_loader))
    scaler = GradScaler(enabled=(cfg.mixed_precision and device == "cuda"))

    run_dir = RESULTS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    writer = SummaryWriter(run_dir / "tb")

    # Hardcode Drive path for checkpoints when running on Colab
    drive_models = "/content/drive/MyDrive/cdt-data/models"
    if os.path.exists("/content/drive/MyDrive/cdt-data"):
        os.makedirs(drive_models, exist_ok=True)
        ckpt_path = Path(drive_models) / f"{run_name}_best.pt"
    else:
        ckpt_path = MODELS_DIR / f"{run_name}_best.pt"
    print(f"[train] checkpoints -> {ckpt_path}")

    best_val_loss = float("inf")
    best_epoch = 0
    patience = 0
    history = []

    for epoch in range(cfg.epochs):
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler,
            device, cfg, epoch, writer,
        )
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device, epoch, writer, tag="val",
        )
        if isinstance(scheduler, ReduceLROnPlateau):
            scheduler.step(val_loss)

        history.append({
            "epoch": epoch,
            "train_loss": tr_loss, "train_acc": tr_acc,
            "val_loss": val_loss, "val_acc": val_acc,
        })
        print(f"[train] ep={epoch:02d} tr_loss={tr_loss:.4f} "
              f"tr_acc={tr_acc:.3f} val_loss={val_loss:.4f} val_acc={val_acc:.3f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience = 0
            save_checkpoint(model, ckpt_path)
        else:
            patience += 1
            if patience >= cfg.early_stopping_patience:
                print(f"[train] early stop at epoch {epoch}")
                break

    with open(run_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)
    writer.close()

    return {
        "run_name": run_name,
        "history": history,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "checkpoint_path": str(ckpt_path),
    }

Overwriting /content/cdt-alzheimer-screening/src/train.py


In [5]:
import shutil, time, importlib
from pathlib import Path
import pandas as pd
import torch

# stage TIFFs locally
LOCAL_DATA_DIR = Path('/content/nhats_local')
LOCAL_DATA_DIR.mkdir(exist_ok=True)
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/cdt-data/nhats_raw')
t0 = time.time()
for rf in sorted(DRIVE_DATA_DIR.iterdir()):
    if not rf.is_dir(): continue
    dst = LOCAL_DATA_DIR / rf.name
    if dst.exists() and len(list(dst.glob('*.tif'))) > 0:
        continue
    print(f"[copy] {rf.name}")
    shutil.copytree(rf, dst, dirs_exist_ok=True)
print(f"staged in {(time.time()-t0)/60:.1f} min")

# rebuild local labels
drive_labels = pd.read_csv('/content/drive/MyDrive/cdt-data/labels.csv')
drive_labels['image_path'] = drive_labels['image_path'].str.replace(
    '/content/drive/MyDrive/cdt-data/nhats_raw/',
    '/content/nhats_local/', regex=False)
drive_labels.to_csv('/content/cdt_labels_local.csv', index=False)

# Reload everything in dependency order
import src.config, src.data, src.augmentation, src.train, src.models, src.evaluate, src.utils
for m in [src.config, src.data, src.augmentation, src.train, src.models, src.evaluate, src.utils]:
    importlib.reload(m)

import src.config as cfg
cfg.LABELS_CSV  = Path('/content/cdt_labels_local.csv')
cfg.MODELS_DIR  = Path('/content/drive/MyDrive/cdt-data/models')
cfg.RESULTS_DIR = Path('/content/drive/MyDrive/cdt-data/results')
cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# CRITICAL: force MODELS_DIR into the train module too
src.train.MODELS_DIR = cfg.MODELS_DIR

# Patch save_checkpoint at the train module level (this is what train.py actually calls)
def save_checkpoint_to_drive(model, path):
    p = Path(path)
    # Always redirect to Drive regardless of input path
    drive_path = cfg.MODELS_DIR / p.name
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), drive_path)

src.train.save_checkpoint = save_checkpoint_to_drive
src.utils.save_checkpoint = save_checkpoint_to_drive

# Verify the patch
import inspect
print("Patched save_checkpoint:")
print(inspect.getsource(src.train.save_checkpoint)[:200])

# splits
from src.data import participant_disjoint_split, compute_class_weights, CDTDataset
from src.config import DataConfig, TrainConfig, AugConfig
from src.augmentation import build_train_transform, build_eval_transform

df = pd.read_csv(cfg.LABELS_CSV)
df = df.rename(columns={'cdt_score': 'label' })[['participant_id', 'label', 'image_path']]
df['participant_id'] = df['participant_id'].astype(str)
data_cfg = DataConfig(batch_size=32, num_workers=6)
train_df, val_df, test_df = participant_disjoint_split(df, data_cfg)
class_weights = compute_class_weights(train_df)

print(f"\n[ok] MODELS_DIR = {cfg.MODELS_DIR}")
print(f"[ok] ready for ablation")

staged in 0.0 min
Patched save_checkpoint:
def save_checkpoint_to_drive(model, path):
    p = Path(path)
    # Always redirect to Drive regardless of input path
    drive_path = cfg.MODELS_DIR / p.name
    drive_path.parent.mkdir(parents=True,
[split] participant-disjoint:
  train: 41,551 clocks from 9,549 participants
  val:   8,928 clocks from 2,046 participants
  test:  8,938 clocks from 2,047 participants

[ok] MODELS_DIR = /content/drive/MyDrive/cdt-data/models
[ok] ready for ablation


In [6]:
!grep -A 3 "Always save to Drive" /content/cdt-alzheimer-screening/src/train.py

In [8]:
import time, json
from torch.utils.data import DataLoader
from src.train import train as train_fn
from src.evaluate import collect_predictions, compute_all_metrics
from src.models import get_model
from src.utils import load_checkpoint, get_device

device = get_device()
ABLATION_RESULTS = {}
ABLATION_BASE = 'vit_b16'   # ablating on best model (ViT)
ABLATION_EPOCHS = 5

def run_ablation(use_aug, freeze_backbone, run_name):
    aug_cfg = AugConfig()
    train_tf = build_train_transform(data_cfg.image_size, aug_cfg, use_aug=use_aug)
    eval_tf  = build_eval_transform(data_cfg.image_size)

    train_ds = CDTDataset(train_df, transform=train_tf)
    val_ds   = CDTDataset(val_df,   transform=eval_tf)
    test_ds  = CDTDataset(test_df,  transform=eval_tf)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                              num_workers=6, pin_memory=True, persistent_workers=True,
                              prefetch_factor=4, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False,
                              num_workers=6, pin_memory=True, persistent_workers=True,
                              prefetch_factor=4)
    test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False,
                              num_workers=6, pin_memory=True, persistent_workers=True,
                              prefetch_factor=4)

    train_cfg = TrainConfig(
        epochs=ABLATION_EPOCHS,
        lr_head=1e-3, lr_backbone=1e-5,
        weight_decay=1e-4, dropout=0.3,
        early_stopping_patience=4, mixed_precision=True,
        lr_scheduler='cosine', optimizer='adamw',
    )
    print(f"\n{'='*70}\nABLATION: {run_name}  (aug={use_aug}, frozen={freeze_backbone})\n{'='*70}")
    t0 = time.time()
    summary = train_fn(
        model_name=ABLATION_BASE,
        train_loader=train_loader, val_loader=val_loader,
        class_weights=class_weights, cfg=train_cfg,
        run_name=run_name, freeze_backbone=freeze_backbone,
    )

    # we try multiple possible checkpoint locations
    candidates = [
        cfg.MODELS_DIR / f'{run_name}_best.pt',
        Path(f'/content/cdt-alzheimer-screening/models/{run_name}_best.pt'),
        Path(summary.get('checkpoint_path', '')),
    ]
    ckpt_path = next((p for p in candidates if p and p.exists()), None)
    if ckpt_path is None:
        print(f"[WARN] no checkpoint found for {run_name}, skipping eval")
        return None
    print(f"[loading {ckpt_path}]")

    model, _, _ = get_model(ABLATION_BASE, freeze_backbone=freeze_backbone, dropout=0.3)
    model = load_checkpoint(model, str(ckpt_path), device=device).to(device)
    y_true, y_pred, y_probs = collect_predictions(model, test_loader, device)
    metrics = compute_all_metrics(y_true, y_pred, y_probs)
    metrics['run_name'] = run_name
    metrics['use_aug']  = use_aug
    metrics['frozen']   = freeze_backbone
    metrics['minutes']  = (time.time() - t0) / 60
    ABLATION_RESULTS[run_name] = metrics

    with open(cfg.RESULTS_DIR / 'ablation_results.json', 'w') as f:
        json.dump(ABLATION_RESULTS, f, indent=2)

    print(f"\n[{run_name}] done in {metrics['minutes']:.1f} min")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:20s} {v:.4f}")

    del model
    torch.cuda.empty_cache()
    return metrics

In [9]:
# the ablation 1/4
run_ablation(use_aug=False, freeze_backbone=True, run_name='abl_frozen_noaug')

# we verify the patch worked
import os
ckpt = '/content/drive/MyDrive/cdt-data/models/abl_frozen_noaug_best.pt'
if os.path.exists(ckpt):
    print(f"\n The Checkpoint was saved to Drive: {os.path.getsize(ckpt)/1e6:.0f} MB")
else:
    print(f"\n The Checkpoint was NOT in Drive — STOP and ping me before continuing")
    !ls /content/cdt-alzheimer-screening/models/abl_*


ABLATION: abl_frozen_noaug  (aug=False, frozen=True)
[train] run=abl_frozen_noaug device=cuda
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 229MB/s]
/content/cdt-alzheimer-screening/src/train.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.mixed_precision and device == "cuda"))


[train] params total=85,803,270 trainable=4,614
[train] checkpoints -> /content/drive/MyDrive/cdt-data/models/abl_frozen_noaug_best.pt


epoch 0 [train]:   0%|          | 0/1298 [00:00<?, ?it/s]/content/cdt-alzheimer-screening/src/train.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.mixed_precision):
/content/cdt-alzheimer-screening/src/train.py:62: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[train] ep=00 tr_loss=1.2803 tr_acc=0.432 val_loss=1.1599 val_acc=0.510


[train] ep=01 tr_loss=1.1998 tr_acc=0.464 val_loss=1.1421 val_acc=0.458


[train] ep=02 tr_loss=1.1637 tr_acc=0.477 val_loss=1.1182 val_acc=0.500


[train] ep=03 tr_loss=1.1366 tr_acc=0.486 val_loss=1.1171 val_acc=0.512


[train] ep=04 tr_loss=1.1174 tr_acc=0.492 val_loss=1.1168 val_acc=0.509
[loading /content/drive/MyDrive/cdt-data/models/abl_frozen_noaug_best.pt]

[abl_frozen_noaug] done in 85.0 min
  accuracy             0.5015
  macro_f1             0.4942
  weighted_f1          0.4944
  macro_precision      0.4700
  macro_recall         0.5502
  quadratic_kappa      0.6670
  macro_auc            0.8489
  minutes              85.0395

 The Checkpoint was saved to Drive: 343 MB


In [10]:
# the ablation 2/4
run_ablation(use_aug=True, freeze_backbone=True, run_name='abl_frozen_aug')


ABLATION: abl_frozen_aug  (aug=True, frozen=True)
[train] run=abl_frozen_aug device=cuda


/content/cdt-alzheimer-screening/src/train.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.mixed_precision and device == "cuda"))


[train] params total=85,803,270 trainable=4,614
[train] checkpoints -> /content/drive/MyDrive/cdt-data/models/abl_frozen_aug_best.pt


epoch 0 [train]:   0%|          | 0/1298 [00:00<?, ?it/s]/content/cdt-alzheimer-screening/src/train.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.mixed_precision):
/content/cdt-alzheimer-screening/src/train.py:62: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[train] ep=00 tr_loss=1.3215 tr_acc=0.416 val_loss=1.2153 val_acc=0.495


[train] ep=01 tr_loss=1.2454 tr_acc=0.451 val_loss=1.2462 val_acc=0.439


[train] ep=02 tr_loss=1.2045 tr_acc=0.459 val_loss=1.2105 val_acc=0.466


[train] ep=03 tr_loss=1.1815 tr_acc=0.468 val_loss=1.2242 val_acc=0.477


[train] ep=04 tr_loss=1.1593 tr_acc=0.473 val_loss=1.2107 val_acc=0.474
[loading /content/drive/MyDrive/cdt-data/models/abl_frozen_aug_best.pt]

[abl_frozen_aug] done in 68.8 min
  accuracy             0.4695
  macro_f1             0.4716
  weighted_f1          0.4342
  macro_precision      0.5022
  macro_recall         0.5011
  quadratic_kappa      0.6395
  macro_auc            0.8463
  minutes              68.8110


{'accuracy': 0.46945625419556947,
 'macro_f1': 0.4715785021275491,
 'weighted_f1': 0.43417635678080424,
 'macro_precision': 0.5022183850589601,
 'macro_recall': 0.501071907691624,
 'quadratic_kappa': 0.6395138622746077,
 'macro_auc': 0.8463052049876181,
 'run_name': 'abl_frozen_aug',
 'use_aug': True,
 'frozen': True,
 'minutes': 68.81102378765742}

In [11]:
# the ablation 3/4
run_ablation(use_aug=False, freeze_backbone=False, run_name='abl_unfrozen_noaug')


ABLATION: abl_unfrozen_noaug  (aug=False, frozen=False)
[train] run=abl_unfrozen_noaug device=cuda


/content/cdt-alzheimer-screening/src/train.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.mixed_precision and device == "cuda"))


[train] params total=85,803,270 trainable=85,803,270
[train] checkpoints -> /content/drive/MyDrive/cdt-data/models/abl_unfrozen_noaug_best.pt


epoch 0 [train]:   0%|          | 0/1298 [00:00<?, ?it/s]/content/cdt-alzheimer-screening/src/train.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.mixed_precision):
/content/cdt-alzheimer-screening/src/train.py:62: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[train] ep=00 tr_loss=1.0648 tr_acc=0.559 val_loss=1.1222 val_acc=0.653


[train] ep=01 tr_loss=0.8404 tr_acc=0.657 val_loss=0.9195 val_acc=0.646


[train] ep=02 tr_loss=0.6739 tr_acc=0.700 val_loss=0.9917 val_acc=0.683


[train] ep=03 tr_loss=0.5304 tr_acc=0.739 val_loss=1.1038 val_acc=0.684


[train] ep=04 tr_loss=0.4528 tr_acc=0.764 val_loss=1.1313 val_acc=0.678
[loading /content/drive/MyDrive/cdt-data/models/abl_unfrozen_noaug_best.pt]

[abl_unfrozen_noaug] done in 65.9 min
  accuracy             0.6457
  macro_f1             0.6227
  weighted_f1          0.6429
  macro_precision      0.6174
  macro_recall         0.6511
  quadratic_kappa      0.7992
  macro_auc            0.9156
  minutes              65.8908


{'accuracy': 0.6456701722980532,
 'macro_f1': 0.6226557433900385,
 'weighted_f1': 0.642943684815586,
 'macro_precision': 0.6174481898643659,
 'macro_recall': 0.6510697098492343,
 'quadratic_kappa': 0.799166678072593,
 'macro_auc': 0.9155739484436398,
 'run_name': 'abl_unfrozen_noaug',
 'use_aug': False,
 'frozen': False,
 'minutes': 65.8907848238945}

In [12]:
# the ablation 4/4
run_ablation(use_aug=True, freeze_backbone=False, run_name='abl_unfrozen_aug')


ABLATION: abl_unfrozen_aug  (aug=True, frozen=False)
[train] run=abl_unfrozen_aug device=cuda


/content/cdt-alzheimer-screening/src/train.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.mixed_precision and device == "cuda"))


[train] params total=85,803,270 trainable=85,803,270
[train] checkpoints -> /content/drive/MyDrive/cdt-data/models/abl_unfrozen_aug_best.pt


epoch 0 [train]:   0%|          | 0/1298 [00:00<?, ?it/s]/content/cdt-alzheimer-screening/src/train.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.mixed_precision):
/content/cdt-alzheimer-screening/src/train.py:62: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[train] ep=00 tr_loss=1.0987 tr_acc=0.539 val_loss=1.1125 val_acc=0.615


[train] ep=01 tr_loss=0.9331 tr_acc=0.620 val_loss=0.8815 val_acc=0.648


[train] ep=02 tr_loss=0.8508 tr_acc=0.654 val_loss=0.8786 val_acc=0.672


[train] ep=03 tr_loss=0.7953 tr_acc=0.669 val_loss=0.8587 val_acc=0.678


[train] ep=04 tr_loss=0.7521 tr_acc=0.680 val_loss=0.8735 val_acc=0.684
[loading /content/drive/MyDrive/cdt-data/models/abl_unfrozen_aug_best.pt]

[abl_unfrozen_aug] done in 68.8 min
  accuracy             0.6760
  macro_f1             0.6555
  weighted_f1          0.6753
  macro_precision      0.6489
  macro_recall         0.6684
  quadratic_kappa      0.8117
  macro_auc            0.9257
  minutes              68.7841


{'accuracy': 0.6759901543969569,
 'macro_f1': 0.6554908425639372,
 'weighted_f1': 0.6753260601904754,
 'macro_precision': 0.6488927741020435,
 'macro_recall': 0.6684228227872656,
 'quadratic_kappa': 0.8116853465199282,
 'macro_auc': 0.9257243651578636,
 'run_name': 'abl_unfrozen_aug',
 'use_aug': True,
 'frozen': False,
 'minutes': 68.78410400946935}

In [13]:
import pandas as pd, json
with open(cfg.RESULTS_DIR / 'ablation_results.json') as f:
    res = json.load(f)

rows = []
for name, m in res.items():
    rows.append({
        'run': name,
        'augmentation':    'on'  if m['use_aug']  else 'off',
        'backbone':        'frozen' if m['frozen'] else 'fine-tuned',
        'accuracy':        m['accuracy'],
        'macro_f1':        m['macro_f1'],
        'quadratic_kappa': m['quadratic_kappa'],
        'macro_auc':       m['macro_auc'],
    })
abl = pd.DataFrame(rows).set_index('run')
abl.to_csv(cfg.RESULTS_DIR / 'ablation_table.csv')
print(abl.round(4).to_string())

                   augmentation    backbone  accuracy  macro_f1  quadratic_kappa  macro_auc
run                                                                                        
abl_frozen_noaug            off      frozen    0.5015    0.4942           0.6670     0.8489
abl_frozen_aug               on      frozen    0.4695    0.4716           0.6395     0.8463
abl_unfrozen_noaug          off  fine-tuned    0.6457    0.6227           0.7992     0.9156
abl_unfrozen_aug             on  fine-tuned    0.6760    0.6555           0.8117     0.9257
